# KPS on InverseBench — posterior metrics, x space and y space

The bench's own metric is a **per-member point error**, minimised by collapsing every
particle onto the posterior mean — so on its own it rewards the failure mode an ensemble
method has. This notebook reports it (`x_member_l2` reproduces `relative l2` exactly) next
to the ensemble scores the forecasting literature uses, in both spaces.

| column | meaning | calibrated value |
|---|---|---|
| `member_l2` | mean per-member relative error — **the bench metric** | — |
| `mean_l2` | relative error of the ensemble **mean** | — |
| `skill` | RMSE of the ensemble mean | — |
| `spread` | ensemble sd, Fortin (2014) `(N+1)/N` corrected | = skill |
| `ssr` | spread / skill | **1.0** |
| `crps`, `crps_n` | **fair** (unbiased) CRPS, raw and / rms(truth) | — |
| `cover90` | truth inside [q05, q95] | **0.90** |
| `rank` | P(member < truth) | **0.5** |

`x_*` is against the ground truth, `y_*` against the observation via `recon_obs`.
CRPS is the **fair** estimator on purpose: the biased one improves with N for free, which
would make a particle sweep meaningless.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('/mnt/home/gandry/kps/InverseBench')
sys.path.insert(0, str(ROOT / 'scripts'))

import matplotlib.pyplot as plt, pandas as pd, numpy as np
import report_metrics as rm

CSV = ROOT / 'scripts' / 'metrics.csv'
RECOMPUTE = False            # True re-scores every result file (minutes)
RUNS = 'gen*'                # 'gen*' is the campaign; '*' takes every historical run

if RECOMPUTE or not CSV.exists():
    df = rm.collect(pattern=RUNS)
    df.to_csv(CSV, index=False)
else:
    df = pd.read_csv(CSV)

df['method'] = df.algo + '/' + df.run
print(f'{len(df)} runs x ids;', df.problem.value_counts().to_dict())

## The table

One row per (problem, method, N), averaged over test instances, with the standard error of
that mean — instance variation is the dominant noise source here (~2.3x seed variation), so
a difference smaller than a couple of `sem` is not a difference.

In [ ]:
X = ['x_member_l2', 'x_mean_l2', 'x_skill', 'x_spread', 'x_ssr',
     'x_crps', 'x_crps_n', 'x_cover90', 'x_rank']
Y = ['y_member_l2', 'y_mean_l2', 'y_ssr', 'y_crps_n', 'y_cover90']
COLS = [c for c in X + Y if c in df]
KEY = ['problem', 'method', 'n']

# MEDIANS, NOT MEANS. A diverged run scores in the thousands and moves a 100-id mean by
# orders of magnitude -- blackhole KPSH has a few -- so the mean stops describing the
# typical instance. `diverged` counts them explicitly rather than letting them hide in an
# average, because how OFTEN a method blows up is itself a result.
g = df.groupby(KEY)
tbl = g[COLS].median()
tbl.insert(0, 'ids', g.size())
tbl.insert(1, 'diverged', g.x_mean_l2.apply(lambda s: int((s > 2).sum())))
display(tbl.round(4))

print('mean and its standard error, over NON-diverged ids only:')
ok = df[df.x_mean_l2 <= 2]
keep = [c for c in ('x_mean_l2', 'x_ssr', 'x_crps_n', 'y_mean_l2') if c in df]
display(pd.concat({'mean': ok.groupby(KEY)[keep].mean(),
                   'sem': ok.groupby(KEY)[keep].sem()}, axis=1).round(4))

## Plots

Top row: whether the cloud is a posterior. Bottom row: accuracy, and whether x-space and
y-space agree. The dotted line in the SSR panels is calibration; below it the ensemble is
over-confident, which is this method's known failure mode.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))

for p, gp in df.groupby('problem'):
    s = gp.groupby('n'); m, e = s[COLS].mean(), s[COLS].sem()
    ax[0,0].errorbar(m.index, m.x_skill, e.x_skill, marker='o', label=f'{p} skill')
    ax[0,0].errorbar(m.index, m.x_spread, e.x_spread, marker='s', ls='--', label=f'{p} spread')
    ax[0,1].errorbar(m.index, m.x_ssr, e.x_ssr, marker='o', label=p)
    ax[0,2].errorbar(m.index, m.x_crps_n, e.x_crps_n, marker='o', label=p)
    ax[1,0].errorbar(m.index, m.x_mean_l2, e.x_mean_l2, marker='o', label=f'{p} mean')
    ax[1,0].errorbar(m.index, m.x_member_l2, e.x_member_l2, marker='s', ls='--',
                     label=f'{p} member (bench)')
    ax[1,1].errorbar(m.index, m.y_ssr, e.y_ssr, marker='o', label=p)
    ax[1,2].scatter(gp.x_mean_l2, gp.y_mean_l2, s=14, alpha=.7, label=p)

titles = [(ax[0,0], 'skill vs spread', 'rmse'), (ax[0,1], 'spread-skill ratio', 'ssr'),
          (ax[0,2], 'fair CRPS / rms(truth)', 'crps_n'),
          (ax[1,0], 'x error: ensemble mean vs member', 'relative l2'),
          (ax[1,1], 'y-space spread-skill ratio', 'ssr'),
          (ax[1,2], 'does y-space agree with x-space?', 'y mean rel l2')]
for a, t, yl in titles:
    a.set_title(t); a.set_ylabel(yl); a.legend(fontsize=7); a.grid(alpha=.3)
    if a is not ax[1,2]:
        a.set_xscale('log', base=2); a.set_xlabel('particles N')
for a in (ax[0,1], ax[1,1]):
    a.axhline(1.0, color='k', ls=':', lw=1)
ax[0,0].set_yscale('log'); ax[0,2].set_yscale('log')
ax[1,2].set_xlabel('x mean rel l2')
plt.tight_layout(); plt.show()

## Paired comparison between two methods

Aggregate means hide the blocking variable that matters. Two methods run on the same test
ids should be compared **per id**: the paired difference isolates the method change instead
of measuring it on top of instance variation.

In [ ]:
A, B = 'KPSH/gen_N512', 'KPSH/gen_N128'      # edit these
METRIC = 'x_mean_l2'

pair = (df[df.method.isin([A, B])]
        .pivot_table(index=['problem', 'id'], columns='method', values=METRIC))
pair = pair.dropna()
PROBLEM = None                              # None pools problems -- usually wrong
if PROBLEM:
    pair = pair.loc[PROBLEM]
pair = pair[(pair < 2).all(axis=1)]         # drop diverged ids from the pairing

if pair.empty:
    print('no ids in common — check the method names against the table above')
else:
    d = pair[A] - pair[B]
    print(f'{METRIC}:  {A} - {B}\n'
          f'  mean {d.mean():+.4f}   sem {d.sem():.4f}   '
          f'wins {int((d < 0).sum())}/{len(d)}   ({len(d)} paired ids)')
    fig, a = plt.subplots(figsize=(4.5, 4.5))
    a.scatter(pair[B], pair[A], s=20)
    lo, hi = pair.values.min(), pair.values.max()
    a.plot([lo, hi], [lo, hi], 'k:', lw=1)
    a.set_xlabel(B); a.set_ylabel(A); a.set_title(f'{METRIC} (below the line = {A} better)')
    a.grid(alpha=.3); plt.tight_layout(); plt.show()

## InverseBench's own problem-specific metrics

Run through the bench's **own evaluator**, so these are its numbers and not a
reimplementation. They need the forward operator, which is why they are a separate pass
(blackhole loads ehtim and its uvfits; allow a minute).

| problem | columns | note |
|---|---|---|
| navier-stokes | `relative l2` | mean over members — identical to `x_member_l2` above |
| inv-scatter | `psnr`, `ssim` | per member on `clip(0,1)`, then averaged |
| blackhole | `cp_chi2`, `camp_chi2` | **best member** (`.min()` over the ensemble) |
| blackhole | `psnr`, `blur_psnr (f=10/15/20)` | maximised over blur factors |

Read the blackhole chi-squares with care: `.min()` over the cloud rewards having *one*
good member, so a wide, badly-centred ensemble can score well on them while failing every
distributional column. Chi-square near 1 is the target (the data are fitted to their error
bars); far below 1 is over-fitting the noise.

In [ ]:
OFFICIAL_CSV = ROOT / 'scripts' / 'metrics_official.csv'
RECOMPUTE_OFFICIAL = False
LIMIT = None                 # e.g. 10 for a quick look; None is every id

if RECOMPUTE_OFFICIAL or not OFFICIAL_CSV.exists():
    off = rm.collect(pattern=RUNS, with_official=True, limit=LIMIT)
    off.to_csv(OFFICIAL_CSV, index=False)
else:
    off = pd.read_csv(OFFICIAL_CSV)

off['method'] = off.algo + '/' + off.run
IB = sorted(c for c in off.columns if c.startswith('ib_'))
print(IB)

In [ ]:
# One table per problem: the bench's columns beside the distributional ones, so a method
# that wins on psnr while its cloud has collapsed is visible in one glance.
SHOW = ['x_member_l2', 'x_mean_l2', 'x_ssr', 'x_crps_n', 'x_cover90', 'y_mean_l2']

for prob, gp in off.groupby('problem'):
    cols = [c for c in IB if gp[c].notna().any()] + [c for c in SHOW if c in gp]
    t = gp.groupby(['method', 'n'])[cols].median()
    t.insert(0, 'ids', gp.groupby(['method', 'n']).size())
    t.insert(1, 'diverged', gp.groupby(['method', 'n']).x_mean_l2.apply(
        lambda s: int((s > 2).sum())))
    print(f'\n=== {prob}   (medians)')
    display(t.round(4))

In [ ]:
# Does the bench metric agree with the distributional one? Where it does not, the bench
# column is being driven by something other than posterior quality.
prim = {'navier-stokes': 'ib_relative l2', 'inv-scatter': 'ib_psnr',
        'blackhole': 'ib_psnr'}

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for a, (prob, gp) in zip(ax, off.groupby('problem')):
    c = prim.get(prob)
    if c not in gp or gp[c].isna().all():
        a.set_visible(False); continue
    ok = gp[gp.x_mean_l2 <= 2]
    for meth, gm in ok.groupby('method'):
        a.scatter(gm[c], gm.x_ssr, s=16, alpha=.7, label=meth)
    a.axhline(1.0, color='k', ls=':', lw=1)
    a.set_xlabel(c); a.set_ylabel('x_ssr (1 = calibrated)')
    a.set_title(prob); a.legend(fontsize=6); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Caveats

- **y space has a floor.** `recon_obs` is the operator applied *with* its observation noise,
  so a perfect reconstruction still differs from `observation` by one noise draw. A y error
  below `sigma_noise*sqrt(dim)/||y||` means the noise is being fitted.
- **The blackhole samples used an isotropic `Sigma_y`**, measured at 24.4 against a median
  channel variance of 0.407 on that operator — the case where the assumption is least
  defensible. Read those rows with that in mind.
- **The bench's blackhole metrics are not in the table.** `cp_chi2`, `camp_chi2` and the
  blur-maximised psnr go through the forward operator, and its chi-squares are a *best member*
  (`.min()` over the ensemble), not an ensemble score. Get them with
  `rm.collect(with_official=True)`, which runs the bench's own evaluator.
- **The NS harness is non-deterministic**: identical runs differ, err sd 0.041 per run. These
  runs were not made under `run_det.py`, so per-id numbers carry that noise.